In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1. CARGAR EL DATASET
# ============================================================
df = pd.read_csv(r"C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed.csv", low_memory=False)


In [2]:
# ============================================================
# 2. DIMENSIONES BÁSICAS
# ============================================================
print("=" * 60)
print("DIMENSIONES DEL DATASET")
print("=" * 60)
print(f"Filas (observaciones):    {df.shape[0]:,}")
print(f"Columnas (variables):     {df.shape[1]}")
print(f"Total de celdas:          {df.shape[0] * df.shape[1]:,}")
print(f"Memoria usada:            {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

DIMENSIONES DEL DATASET
Filas (observaciones):    11,840
Columnas (variables):     46
Total de celdas:          544,640
Memoria usada:            8.73 MB


In [3]:
# ============================================================
# 3. TIPOS DE DATOS
# ============================================================
print("\n" + "=" * 60)
print("TIPOS DE DATOS")
print("=" * 60)
print(df.dtypes.value_counts())

# Detalle por tipo
for dtype in df.dtypes.value_counts().index:
    cols = df.select_dtypes(include=[dtype]).columns.tolist()
    print(f"\n{dtype} ({len(cols)} columnas):")
    for col in cols:
        print(f"   - {col}")


TIPOS DE DATOS
float64    27
int64      17
object      2
Name: count, dtype: int64

float64 (27 columnas):
   - lastSoldPrice_hpi_adjusted
   - log_price
   - bedrooms
   - bathrooms
   - livingArea
   - yearBuilt
   - latitude
   - longitude
   - lotAreaValue
   - taxAssessedValue
   - propertyTaxRate
   - latest_tax_value
   - latest_tax_paid
   - num_tax_records
   - num_sales
   - num_price_changes
   - last_listing_price
   - avg_school_rating
   - max_school_rating
   - num_nearby_schools
   - min_school_distance
   - has_hoa
   - hoa_fee_monthly
   - property_age
   - bath_to_bed_ratio
   - log_living_area
   - log_lot_area

int64 (17 columnas):
   - zpid
   - photoCount
   - zipcode
   - has_pool
   - has_garage
   - has_waterfront
   - tag_price_cut
   - tag_new_construction
   - tag_foreclosure
   - zip_3digit
   - desc_length
   - desc_word_count
   - desc_is_boilerplate
   - desc_mentions_renovated
   - desc_mentions_pool
   - desc_mentions_view
   - desc_mentions_new

obj

In [4]:
# ============================================================
# 4. RESUMEN COMPLETO DE COLUMNAS
# ============================================================
print("\n" + "=" * 60)
print("RESUMEN DE TODAS LAS COLUMNAS")
print("=" * 60)

resumen = pd.DataFrame({
    'Columna': df.columns,
    'Tipo': df.dtypes.values,
    'No_Nulos': df.count().values,
    'Nulos': df.isnull().sum().values,
    'Pct_Nulos': (df.isnull().sum().values / len(df) * 100).round(2),
    'Valores_Unicos': [df[col].nunique() for col in df.columns]
})
print(resumen.to_string(index=False))


RESUMEN DE TODAS LAS COLUMNAS
                   Columna    Tipo  No_Nulos  Nulos  Pct_Nulos  Valores_Unicos
                      zpid   int64     11840      0       0.00           11840
lastSoldPrice_hpi_adjusted float64     11840      0       0.00            9924
                 log_price float64     11840      0       0.00            9922
               description  object     11840      0       0.00           11804
                  bedrooms float64     11360    480       4.05              11
                 bathrooms float64     11615    225       1.90              20
                livingArea float64     11668    172       1.45            2378
                 yearBuilt float64     11529    311       2.63             110
                  latitude float64     11809     31       0.26            8610
                 longitude float64     11809     31       0.26            7442
              lotAreaValue float64      6494   5346      45.15            2083
                photo

In [5]:
# ============================================================
# 5. IDENTIFICAR VARIABLE OBJETIVO
# ============================================================
print("\n" + "=" * 60)
print("VARIABLE OBJETIVO (TARGET)")
print("=" * 60)
targets = ['lastSoldPrice_hpi_adjusted', 'log_price']
for col in targets:
    if col in df.columns:
        print(f"{col}:")
        print(f"   Min:   {df[col].min():.2f}")
        print(f"   Max:   {df[col].max():.2f}")
        print(f"   Media: {df[col].mean():.2f}")
        print(f"   Std:   {df[col].std():.2f}")


VARIABLE OBJETIVO (TARGET)
lastSoldPrice_hpi_adjusted:
   Min:   51275.85
   Max:   1986843.24
   Media: 559019.36
   Std:   350971.43
log_price:
   Min:   10.84
   Max:   14.50
   Media: 13.03
   Std:   0.66


In [6]:
# ============================================================
# 6. DETECTAR PROBLEMAS PARA REGRESIÓN LINEAL
# ============================================================
print("\n" + "=" * 60)
print("PROBLEMAS CRÍTICOS PARA REGRESIÓN LINEAL")
print("=" * 60)

# Columnas con nulos
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
problemas = pd.DataFrame({
    'Columna': nulos.index,
    'Nulos': nulos.values,
    'Pct_Nulos': nulos_pct.values
})
problemas = problemas[problemas['Nulos'] > 0].sort_values('Pct_Nulos', ascending=False)
print("\nColumnas con valores faltantes:")
print(problemas.to_string(index=False))

# Columnas constantes (varianza cero)
print("\nColumnas constantes o casi constantes:")
for col in df.columns:
    if df[col].nunique() <= 1:
        print(f"   ⚠️  {col}: {df[col].nunique()} valor único")

# Columnas binarias
binarias = [col for col in df.columns if df[col].nunique() == 2]
print(f"\nColumnas binarias ({len(binarias)}):")
for col in binarias:
    print(f"   - {col}: {df[col].value_counts().to_dict()}")


PROBLEMAS CRÍTICOS PARA REGRESIÓN LINEAL

Columnas con valores faltantes:
           Columna  Nulos  Pct_Nulos
      lotAreaValue   5346      45.15
      log_lot_area   5346      45.15
last_listing_price   3925      33.15
 bath_to_bed_ratio    582       4.92
   latest_tax_paid    539       4.55
  taxAssessedValue    524       4.43
  latest_tax_value    525       4.43
          bedrooms    480       4.05
         yearBuilt    311       2.63
      property_age    311       2.63
         bathrooms    225       1.90
        livingArea    172       1.45
   log_living_area    172       1.45
         longitude     31       0.26
          latitude     31       0.26
   propertyTaxRate      2       0.02

Columnas constantes o casi constantes:
   ⚠️  tag_price_cut: 1 valor único

Columnas binarias (12):
   - num_nearby_schools: {3.0: 11817, 2.0: 23}
   - has_hoa: {1.0: 6142, 0.0: 5698}
   - has_pool: {0: 9609, 1: 2231}
   - has_garage: {0: 11039, 1: 801}
   - has_waterfront: {0: 11169, 1: 671}
 